In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
from memvp.build import create_model
import torch
from dataclasses import dataclass
from memvp.tokenizer import Tokenizer
import json
from PIL import Image
from torchvision.transforms import transforms as T
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
import warnings
warnings.filterwarnings("ignore")

transform  = T.Compose(
            [T.Resize((224, 224), interpolation=Image.BICUBIC), 
             T.ToTensor(),
             T.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)])

@dataclass
class ModelArgs_7B:
    llama_model_path = './data/weights/'
    llm_model = '7B'
    max_seq_len = 512
    hidden_proj = 128
    emb = 320
    cpu_load = False
    adapter_scale = 0.1
    adapter_dim = 12
    gradient_checkpointing = False
    is_train = False
    data_root = './data/'
    adapter_path='./ckpts'
    image_root='/ai/teacher/dkc/Assets/GQA/images'

args = ModelArgs_7B()
llama = create_model(args)
adapter = torch.load('/ai/teacher/dkc/sub/MemVP-G/MemVP-GQA9M-single/checkpoint-1.pth')['model']
sd = {}
for k in adapter:
    sd[k.replace('module.', '')] = adapter[k]
llama.load_state_dict(sd, False)

tokenizer = Tokenizer(model_path=os.path.join(args.llama_model_path, 'tokenizer.model'))


/root/miniconda3/envs/dkc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using model path: ./data/weights/, model_name: 7B
['adapter_emb1', 'adapter_emb2', 'backbone.visual.transformer.resblocks.0.adapter_mlp.fc1.weight', 'backbone.visual.transformer.resblocks.0.adapter_mlp.fc2.weight', 'backbone.visual.transformer.resblocks.1.adapter_mlp.fc1.weight', 'backbone.visual.transformer.resblocks.1.adapter_mlp.fc2.weight', 'backbone.visual.transformer.resblocks.2.adapter_mlp.fc1.weight', 'backbone.visual.transformer.resblocks.2.adapter_mlp.fc2.weight', 'backbone.visual.transformer.resblocks.3.adapter_mlp.fc1.weight', 'backbone.visual.transformer.resblocks.3.adapter_mlp.fc2.weight', 'backbone.visual.transformer.resblocks.4.adapter_mlp.fc1.weight', 'backbone.visual.transformer.resblocks.4.adapter_mlp.fc2.weight', 'backbone.visual.transformer.resblocks.5.adapter_mlp.fc1.weight', 'backbone.visual.transformer.resblocks.5.adapter_mlp.fc2.weight', 'backbone.visual.transformer.resblocks.6.adapter_mlp.fc1.weight', 'backbone.visual.transformer.resblocks.6.adapter_mlp.fc2.we

In [12]:
img_id = '2331963'
img = transform(Image.open(f'{args.image_root}/{img_id}.jpg').convert('RGB')).unsqueeze(0)
img = transform(Image.open(f'/ai/teacher/dkc/sub/MemVP-G/xldesktop.jpg').convert('RGB')).unsqueeze(0)
llama.generate(["Question: Can you see an elephant?\nRespond:The answer is"], img, [1], 20, tokenizer)

['no']